# GPT-5.6 retained reasoning: `current_turn` vs `all_turns`

This paired experiment compares `current_turn + previous_response_id` with
`all_turns + previous_response_id` on an identifiable hidden-policy workload.
It separates labeled learning, labeled boundary calibration, and a feedback-free
blind evaluation so that evaluation performance cannot improve from test-time
correctness feedback.

Paid execution is opt-in. By default, **Run All** performs the free preflight.
Every completed `(trial, policy profile, arm)` unit is persisted atomically and
can be resumed without restarting the kernel.


## Workload: learn, disambiguate, then generalize without feedback

- **Situation:** An engineer processes tickets for the fictional `OrchidDesk`
  service under an undocumented support policy.
- **Phase 1 — learning (10 tickets):** Two isolated labeled examples for each
  latent rule establish the action codebook.
- **Phase 2 — boundary calibration (8 tickets):** Labeled counterfactual cases
  reveal conjunction boundaries and precedence between simultaneously active
  rules.
- **Phase 3 — blind evaluation (20 tickets):** Balanced held-out cases return
  only `recorded=true`; correctness and accepted actions remain hidden.
- **Actions:** `monitor`, `retry`, `workaround`, `escalate`, or `rollback`.
- **Synthetic boundary:** No customer or production system is contacted.

The evaluation contains four cases per decisive rule and four expected uses of
each action. Policy profiles use balanced arbitrary action mappings, preventing
ordinary support intuition from revealing the answer.


## 1. Experimental question and leakage controls

The experiment asks two separate questions:

1. Does retained reasoning improve strategy transfer on feedback-free cases?
2. Does it reduce decision-time reasoning while quality remains acceptable?

Leakage and ambiguity controls:

- the solver never receives rule names, precedence, policy mappings, hidden
  grades, or pass thresholds;
- labeled phases always return the accepted action, regardless of whether the
  submitted action was correct, so both arms receive the same supervision;
- evaluation feedback never reveals correctness;
- tool arguments contain only ticket IDs and action enums—no visible rationale
  or external scratchpad;
- a free preflight enumerates 2,880 candidate mapping/precedence combinations
  and requires exactly one behavior to match the 18 labeled cases;
- evaluation cases and order are identical within each pair, while phase order
  is deterministically shuffled across profile and trial.


## 2. Setup, Fast processing, and paid-mode guard

`SERVICE_TIER="fast"` requests Fast processing. The API may report the effective
tier as `priority`; requested and effective values are both saved. Set the tier
to `None` to use the project default.

Set `RUN_PAIRED_PILOT=True` and use **Run All** to start a paid pilot. A full
5×3 paired pilot contains 30 run units and 38 tickets per unit, so review the
preflight before enabling it. To resume an interrupted run, keep the exact same
contract and set `PILOT_RESUME_PATH="latest"` or an explicit result path.

An output-token limit is part of the experiment contract. Completed units from
a different workload or token limit must not be mixed into this artifact.


In [1]:
import json
import random
import traceback
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from statistics import median
from time import perf_counter

from agents.tracing import custom_span, flush_traces, response_span, trace
from openai import OpenAI

try:
    from responses_lab.support_policy_v2 import (
        ACTIONS,
        HARNESS_VERSION,
        SupportPolicyV2Simulator,
        scenario_ids,
        stable_json_sha256,
        ticket_sequence,
        validate_workload,
    )
    from responses_lab.tracing import configure_tracing
    from responses_lab.utils import load_api_key
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Run `uv sync` from the project root and select the project's .venv kernel."
    ) from exc

# Paid execution is opt-in.
RUN_PAIRED_PILOT = False
PILOT_RESUME_PATH = None  # None, "latest", or an explicit artifact path.

MODEL = "gpt-5.6"
REASONING_EFFORT = "medium"
SERVICE_TIER = "fast"  # Set to None to use the project default processing tier.
TRIALS = 3
RANDOM_SEED = 56
MINIMUM_EVALUATION_ACCURACY = 0.70
MINIMUM_STRATEGY_CONSISTENCY = 0.80
MAX_TOOL_OUTPUT_TOKENS = 1_500
MAX_FINAL_OUTPUT_TOKENS = 300

ARM_CONFIGS = [
    {"name": "current_turn", "reasoning_context": "current_turn"},
    {"name": "all_turns", "reasoning_context": "all_turns"},
]
BENCHMARK_SCENARIOS = scenario_ids()

if SERVICE_TIER not in {None, "fast"}:
    raise ValueError("This notebook intentionally supports only None or 'fast'.")

def find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "pyproject.toml").is_file() and (base / "notebooks").is_dir():
            return base
    raise FileNotFoundError("Could not locate the response-api-examples project root.")

PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "02_retained_reasoning"
PRICING_PATH = PROJECT_ROOT / "pricing" / "openai-model-pricing-2026-08-25.json"
pricing_snapshot = json.loads(PRICING_PATH.read_text(encoding="utf-8"))

client = OpenAI(api_key=load_api_key(), max_retries=3) if RUN_PAIRED_PILOT else None
tracing_setup = configure_tracing()
print("Paid paired pilot:", RUN_PAIRED_PILOT)
print("Model / reasoning:", MODEL, REASONING_EFFORT)
print("Requested processing tier:", SERVICE_TIER or "project default")
print("Resume path:", PILOT_RESUME_PATH)
print("Local trace file:", tracing_setup.local_path)
print("OpenAI dashboard export:", tracing_setup.openai_dashboard_enabled)


Paid paired pilot: True
Model / reasoning: gpt-5.6 medium
Requested processing tier: fast
Resume path: None
Local trace file: traces/local_traces.jsonl
OpenAI dashboard export: False


## 3. Arms, controls, and predeclared outcomes

| Arm | Continuation | Prior reasoning available |
|---|---|---|
| `current_turn` | `store=True` + `previous_response_id` | Observable conversation state remains, but older reasoning is not retained |
| `all_turns` | `store=True` + `previous_response_id` | Available reasoning items from earlier turns can be reused |

Primary quality outcome: accuracy over 20 blind evaluation tickets. Primary
strategy outcome: rule-level consistency with acceptable accuracy. Primary
thinking outcome: reasoning tokens generated for evaluation dispositions.
Latency is secondary. Standard-rate cost estimates are directional and are not
a Fast Processing billing statement.


In [2]:
preflight = validate_workload()
assert preflight["scenario_count"] == len(BENCHMARK_SCENARIOS) == 5
for summary in preflight["scenarios"].values():
    assert summary["phase_counts"] == {
        "learning": 10, "boundary": 8, "evaluation": 20,
    }
    assert set(summary["evaluation_rule_counts"].values()) == {4}
    assert set(summary["evaluation_action_counts"].values()) == {4}
    assert summary["identifiability"]["candidate_parameterizations"] == 2_880
    assert summary["identifiability"]["matching_behavior_signatures"] == 1
    assert summary["identifiability"]["identifiable"] is True
assert {arm["reasoning_context"] for arm in ARM_CONFIGS} == {
    "current_turn", "all_turns"
}

# Exercise a complete local run and verify that evaluation feedback is blind.
sample = SupportPolicyV2Simulator(BENCHMARK_SCENARIOS[0])
evaluation_feedback = []
for ticket in sample.tickets:
    sample.inspect_ticket(ticket["ticket_id"])
    feedback = sample.submit_disposition(ticket["ticket_id"], ACTIONS[0])
    if ticket["phase"] == "evaluation":
        evaluation_feedback.append(feedback)
sample_grade = sample.grade()
assert sample_grade["complete"] is True
assert sample_grade["completed_ticket_count"] == 38
assert all(set(row) == {"ticket_id", "recorded"} for row in evaluation_feedback)
print(json.dumps(preflight, indent=2))
print("Free local preflight passed: workload is balanced, identifiable, and blind.")


{
  "harness_version": "support-policy-v2-retained-reasoning-v1",
  "scenario_count": 5,
  "scenarios": {
    "POL2-401": {
      "phase_counts": {
        "learning": 10,
        "boundary": 8,
        "evaluation": 20
      },
      "evaluation_rule_counts": {
        "dependency_degraded": 4,
        "repeat_without_workaround": 4,
        "enterprise_freeze": 4,
        "workaround_available": 4,
        "default": 4
      },
      "evaluation_action_counts": {
        "monitor": 4,
        "retry": 4,
        "workaround": 4,
        "escalate": 4,
        "rollback": 4
      },
      "identifiability": {
        "scenario_id": "POL2-401",
        "labeled_case_count": 18,
        "candidate_parameterizations": 2880,
        "matching_parameterizations": 1,
        "matching_behavior_signatures": 1,
        "identifiable": true
      }
    },
    "POL2-402": {
      "phase_counts": {
        "learning": 10,
        "boundary": 8,
        "evaluation": 20
      },
      "evaluation

## 4. Solver-visible contract

The harness exposes exactly one tool at a time: inspect the current ticket, then
submit one disposition. Labeled-phase feedback supplies only the accepted action
and never exposes the latent rule. Blind-evaluation feedback confirms recording
without revealing correctness. Instructions are repeated because instructions
from a prior response are not automatically carried through
`previous_response_id`.


In [3]:
INSPECT_TOOL = {
    "type": "function",
    "name": "inspect_ticket",
    "description": "Return observable fields for the specified OrchidDesk ticket.",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {"ticket_id": {"type": "string"}},
        "required": ["ticket_id"],
        "additionalProperties": False,
    },
}

DISPOSITION_TOOL = {
    "type": "function",
    "name": "submit_disposition",
    "description": "Submit exactly one policy disposition for the current ticket.",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string"},
            "action": {"type": "string", "enum": list(ACTIONS)},
        },
        "required": ["ticket_id", "action"],
        "additionalProperties": False,
    },
}

FINAL_FORMAT = {
    "type": "json_schema",
    "name": "support_policy_run_complete",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "status": {"type": "string", "enum": ["complete"]},
            "processed_tickets": {"type": "integer"},
        },
        "required": ["status", "processed_tickets"],
        "additionalProperties": False,
    },
}

STABLE_INSTRUCTIONS = """You operate the synthetic OrchidDesk support queue.
Use only the tool currently available. Inspect the specified ticket, then choose
exactly one disposition from the tool enum. Learn the undocumented policy only
from observable ticket fields and labeled-phase feedback. Evaluation feedback
contains no correctness signal. The policy stays fixed for this run. Do not emit a policy summary, rationale, hypothesis list, or other
external scratchpad. Do not process a different ticket."""


## 5. Stateful runner and Trace capture

Every response is stored and the next call uses its `response.id`. Each phase is
shuffled deterministically from `(seed, profile, trial)`, producing the same
order for paired arms without making one fixed position synonymous with a rule.
Requested/effective reasoning context, service tier, token usage, timing,
linkage, response status, and standard-rate cost are captured per call.


In [4]:
def item_type(item):
    return item.get("type", item.get("role", "unknown")) if isinstance(item, dict) else item.type

def usage_metrics(usage):
    if usage is None:
        return {key: 0 for key in (
            "input_tokens", "cached_tokens", "cache_write_tokens",
            "output_tokens", "reasoning_tokens",
        )}
    input_details = getattr(usage, "input_tokens_details", None)
    output_details = getattr(usage, "output_tokens_details", None)
    return {
        "input_tokens": usage.input_tokens,
        "cached_tokens": getattr(input_details, "cached_tokens", 0) or 0,
        "cache_write_tokens": getattr(input_details, "cache_write_tokens", 0) or 0,
        "output_tokens": usage.output_tokens,
        "reasoning_tokens": getattr(output_details, "reasoning_tokens", 0) or 0,
    }

def estimate_standard_rate_cost(metrics):
    priced_model = pricing_snapshot["aliases"].get(MODEL, MODEL)
    prices = pricing_snapshot["models"][priced_model]
    uncached = metrics["input_tokens"] - metrics["cached_tokens"] - metrics["cache_write_tokens"]
    if uncached < 0:
        raise ValueError("Cache token categories exceed input tokens.")
    return (
        uncached * prices["input"]
        + metrics["cached_tokens"] * prices["cached_input"]
        + metrics["cache_write_tokens"] * prices["cache_write_input"]
        + metrics["output_tokens"] * prices["output"]
    ) / 1_000_000

class SolverResponseContractError(RuntimeError):
    def __init__(self, message, response):
        super().__init__(message)
        self.solver_response_id = getattr(response, "id", None)
        self.solver_response_status = getattr(response, "status", None)
        details = getattr(response, "incomplete_details", None)
        self.solver_incomplete_reason = getattr(details, "reason", None)

def require_completed_response(response, stage):
    if response.status == "completed":
        return
    details = getattr(response, "incomplete_details", None)
    reason = getattr(details, "reason", None) or "unknown_reason"
    raise SolverResponseContractError(
        f"{stage} response {response.id} is {response.status}: {reason}. "
        f"Increase the matching output-token limit or resume this run unit.",
        response,
    )

def parse_tool_call(response, expected_name):
    require_completed_response(response, expected_name)
    calls = [item for item in response.output if item.type == "function_call"]
    if len(calls) != 1 or calls[0].name != expected_name:
        raise SolverResponseContractError(
            f"Expected one {expected_name} call, got "
            f"{[(call.name, call.call_id) for call in calls]}",
            response,
        )
    call = calls[0]
    raw_arguments = (call.arguments or "").strip()
    if not raw_arguments:
        raise SolverResponseContractError(
            f"{expected_name} response {response.id} has empty arguments.",
            response,
        )
    try:
        arguments = json.loads(raw_arguments)
    except json.JSONDecodeError as exc:
        raise SolverResponseContractError(
            f"{expected_name} response {response.id} has invalid JSON arguments: {exc}",
            response,
        ) from exc
    if not isinstance(arguments, dict):
        raise SolverResponseContractError(
            f"{expected_name} arguments must be a JSON object.", response
        )
    return call, arguments

def parse_final_response(response):
    require_completed_response(response, "final")
    try:
        payload = json.loads(response.output_text)
    except json.JSONDecodeError as exc:
        raise SolverResponseContractError(
            f"Final response {response.id} has invalid JSON: {exc}", response
        ) from exc
    if not isinstance(payload, dict):
        raise SolverResponseContractError(
            f"Final response {response.id} must be a JSON object.", response
        )
    return payload

def call_solver(*, arm, trial, scenario_id, step, phase, input_items,
                previous_response_id, tools, final=False):
    request = {
        "model": MODEL,
        "instructions": STABLE_INSTRUCTIONS,
        "input": input_items,
        "parallel_tool_calls": False,
        "reasoning": {
            "effort": REASONING_EFFORT,
            "context": arm["reasoning_context"],
        },
        "text": {
            "verbosity": "low",
            "format": FINAL_FORMAT if final else {"type": "text"},
        },
        "max_output_tokens": (
            MAX_FINAL_OUTPUT_TOKENS if final else MAX_TOOL_OUTPUT_TOKENS
        ),
        "prompt_cache_key": f"02b-v2:{scenario_id}:{trial}:{arm['name']}",
        "store": True,
    }
    if tools:
        request.update({"tools": tools, "tool_choice": "required"})
    if previous_response_id is not None:
        request["previous_response_id"] = previous_response_id

    span_data = {
        "arm": arm["name"], "trial": trial, "scenario_id": scenario_id,
        "step": step, "phase": phase,
        "requested_reasoning_context": arm["reasoning_context"],
        "requested_service_tier": SERVICE_TIER,
        "has_previous_response_id": previous_response_id is not None,
        "new_input_item_counts": dict(Counter(item_type(item) for item in input_items)),
    }
    with custom_span("responses.support_policy_v2.solver_call", data=span_data):
        with response_span() as api_span:
            started = perf_counter()
            if SERVICE_TIER is None:
                response = client.responses.create(**request)
            else:
                response = client.responses.create(service_tier=SERVICE_TIER, **request)
            latency_ms = round((perf_counter() - started) * 1000, 1)
            api_span.span_data.response = response
            api_span.span_data.usage = response.usage.model_dump() if response.usage else None
    metrics = usage_metrics(response.usage)
    row = {
        **span_data,
        "effective_reasoning_context": getattr(response.reasoning, "context", None),
        "effective_service_tier": getattr(response, "service_tier", None),
        "previous_response_id": previous_response_id,
        "response_previous_response_id": response.previous_response_id,
        "response_id": response.id,
        "response_status": response.status,
        "incomplete_reason": getattr(
            getattr(response, "incomplete_details", None), "reason", None
        ),
        "max_output_tokens": request["max_output_tokens"],
        "output_types": [item.type for item in response.output],
        "latency_ms": latency_ms,
        **metrics,
        "standard_rate_cost_estimate_usd": estimate_standard_rate_cost(metrics),
    }
    span_data.update(row)
    return response, row

def tool_output(call_id, payload):
    return {
        "type": "function_call_output",
        "call_id": call_id,
        "output": json.dumps(payload, ensure_ascii=False),
    }

def ordered_tickets(tickets, scenario_id, trial):
    groups = {
        phase: [ticket for ticket in tickets if ticket["phase"] == phase]
        for phase in ("learning", "boundary", "evaluation")
    }
    rng = random.Random(f"{RANDOM_SEED}:{scenario_id}:{trial}")
    ordered = []
    for phase in ("learning", "boundary", "evaluation"):
        rng.shuffle(groups[phase])
        ordered.extend(groups[phase])
    return ordered

def run_scenario(arm, scenario_id, trial):
    simulator = SupportPolicyV2Simulator(scenario_id)
    tickets = ordered_tickets(simulator.tickets, scenario_id, trial)
    previous_response_id = None
    pending_output = None
    rows = []
    case_rows = []
    with trace(
        f"02b v2 support-policy {scenario_id} trial {trial}: {arm['name']}",
        metadata={
            "model": MODEL, "arm": arm["name"], "scenario_id": scenario_id,
            "trial": trial, "store": True, "service_tier": SERVICE_TIER,
        },
    ) as workflow_trace:
        for index, ticket in enumerate(tickets, start=1):
            ticket_id = ticket["ticket_id"]
            phase = ticket["phase"]
            user_item = {
                "role": "user",
                "content": json.dumps({
                    "queue": "OrchidDesk", "ticket_id": ticket_id,
                    "phase": phase,
                    "instruction": "Inspect and process this ticket under the unchanged policy.",
                }),
            }
            inspect_input = [user_item] if pending_output is None else [pending_output, user_item]
            inspect_response, inspect_row = call_solver(
                arm=arm, trial=trial, scenario_id=scenario_id,
                step=f"{ticket_id}:inspect", phase=phase,
                input_items=inspect_input,
                previous_response_id=previous_response_id,
                tools=[INSPECT_TOOL],
            )
            rows.append(inspect_row)
            inspect_call, inspect_args = parse_tool_call(
                inspect_response, "inspect_ticket"
            )
            if inspect_args["ticket_id"] != ticket_id:
                raise RuntimeError(f"Solver inspected {inspect_args['ticket_id']} instead of {ticket_id}")
            observation = simulator.inspect_ticket(ticket_id)

            decision_response, decision_row = call_solver(
                arm=arm, trial=trial, scenario_id=scenario_id,
                step=f"{ticket_id}:decision", phase=phase,
                input_items=[tool_output(inspect_call.call_id, observation)],
                previous_response_id=inspect_response.id,
                tools=[DISPOSITION_TOOL],
            )
            rows.append(decision_row)
            decision_call, decision_args = parse_tool_call(
                decision_response, "submit_disposition"
            )
            if decision_args["ticket_id"] != ticket_id:
                raise RuntimeError(f"Solver decided {decision_args['ticket_id']} instead of {ticket_id}")
            feedback = simulator.submit_disposition(ticket_id, decision_args["action"])
            hidden_grade = simulator.decisions[-1]
            case_rows.append({
                "ticket_id": ticket_id,
                "phase": phase,
                "action": decision_args["action"],
                "accepted": hidden_grade["accepted"],
                "decisive_rule": hidden_grade["decisive_rule"],
                "expected_action": hidden_grade["expected_action"],
                "inspect_reasoning_tokens": inspect_row["reasoning_tokens"],
                "decision_reasoning_tokens": decision_row["reasoning_tokens"],
                "inspect_latency_ms": inspect_row["latency_ms"],
                "decision_latency_ms": decision_row["latency_ms"],
            })
            previous_response_id = decision_response.id
            pending_output = tool_output(decision_call.call_id, feedback)

        final_response, final_row = call_solver(
            arm=arm, trial=trial, scenario_id=scenario_id,
            step="final", phase="final",
            input_items=[
                pending_output,
                {"role": "user", "content": json.dumps({
                    "instruction": "Return only the required completion JSON.",
                    "processed_tickets": len(tickets),
                })},
            ],
            previous_response_id=previous_response_id,
            tools=[], final=True,
        )
        rows.append(final_row)
        final_json = parse_final_response(final_response)

    grade = simulator.grade()
    context_ok = all(
        row["effective_reasoning_context"] == row["requested_reasoning_context"]
        for row in rows
    )
    linkage_ok = all(
        row["previous_response_id"] == row["response_previous_response_id"]
        for row in rows
    )
    return {
        "arm": arm["name"], "scenario_id": scenario_id, "trial": trial,
        "trace_id": workflow_trace.trace_id,
        "requested_service_tier": SERVICE_TIER,
        "effective_service_tiers": sorted({
            row["effective_service_tier"] for row in rows
            if row["effective_service_tier"] is not None
        }),
        "context_ok": context_ok, "linkage_ok": linkage_ok,
        "protocol_success": final_json == {"status": "complete", "processed_tickets": len(tickets)},
        "final_json": final_json,
        "grade": grade,
        "case_rows": case_rows,
        "response_rows": rows,
        "total_reasoning_tokens": sum(row["reasoning_tokens"] for row in rows),
        "total_latency_ms": sum(row["latency_ms"] for row in rows),
        "standard_rate_cost_estimate_usd": sum(
            row["standard_rate_cost_estimate_usd"] for row in rows
        ),
    }


## 6. Versioned result persistence and resume

The artifact is atomically replaced after every complete run unit. On failure,
completed units remain in the result JSON and the failing unit is recorded in a
JSONL log. Resume restarts only the incomplete unit from its first learning
ticket, avoiding partially learned state. Workload hashes, phase sizes, token
limits, and analysis thresholds are part of the compatibility contract.


In [5]:
def atomic_write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    temporary.replace(path)

def append_failure_log(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")

def experiment_contract():
    return {
        "harness_version": HARNESS_VERSION,
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
        "requested_service_tier": SERVICE_TIER,
        "trials": TRIALS,
        "random_seed": RANDOM_SEED,
        "scenario_ids": BENCHMARK_SCENARIOS,
        "arms": ARM_CONFIGS,
        "ticket_sequence_hashes": {
            scenario_id: stable_json_sha256(ticket_sequence(scenario_id))
            for scenario_id in BENCHMARK_SCENARIOS
        },
        "pricing_snapshot_date": pricing_snapshot["snapshot_date"],
        "phase_counts": {"learning": 10, "boundary": 8, "evaluation": 20},
        "minimum_evaluation_accuracy": MINIMUM_EVALUATION_ACCURACY,
        "minimum_strategy_consistency": MINIMUM_STRATEGY_CONSISTENCY,
        "max_tool_output_tokens": MAX_TOOL_OUTPUT_TOKENS,
        "max_final_output_tokens": MAX_FINAL_OUTPUT_TOKENS,
    }

def resolve_resume_path():
    if PILOT_RESUME_PATH is None:
        return None
    if PILOT_RESUME_PATH == "latest":
        candidates = sorted(ARTIFACT_DIR.glob("results-*.json"))
        if not candidates:
            raise FileNotFoundError("No 02b v2 result artifact is available to resume.")
        return candidates[-1]
    candidate = Path(PILOT_RESUME_PATH).expanduser()
    if not candidate.is_absolute():
        candidate = PROJECT_ROOT / candidate
    if not candidate.is_file():
        raise FileNotFoundError(candidate)
    return candidate

def validate_resume_payload(payload):
    if payload.get("artifact_schema_version") != 1:
        raise ValueError("Unsupported artifact schema version.")
    if payload.get("experiment_contract") != experiment_contract():
        raise ValueError("Resume artifact does not match the current experiment contract.")
    results = payload.get("results")
    if not isinstance(results, list):
        raise ValueError("Resume artifact results must be a list.")
    keys = [(row["trial"], row["scenario_id"], row["arm"]) for row in results]
    if len(keys) != len(set(keys)):
        raise ValueError("Resume artifact contains duplicate run units.")
    return results

def persist_results(results, *, path, run_id, created_at, status, failure_log_path):
    payload = {
        "artifact_schema_version": 1,
        "run_id": run_id,
        "created_at": created_at,
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "status": status,
        "analysis_complete": False,
        "expected_run_count": TRIALS * len(BENCHMARK_SCENARIOS) * len(ARM_CONFIGS),
        "completed_run_count": len(results),
        "experiment_contract": experiment_contract(),
        "failure_log_path": str(failure_log_path),
        "results": results,
    }
    atomic_write_json(path, payload)
    return payload


## 7. Resumable paired pilot

Arm order is deterministically counterbalanced within each trial and profile.
Each complete unit is saved immediately. An interrupted unit is excluded until
it is rerun from the start. If an employee-key access error occurs, restore the
approved network path and resume the same compatible artifact.


In [6]:
def run_paired_pilot():
    resume_path = resolve_resume_path()
    if resume_path is None:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        created_at = datetime.now(timezone.utc).isoformat()
        result_path = ARTIFACT_DIR / f"results-{run_id}.json"
        failure_log_path = ARTIFACT_DIR / f"failures-{run_id}.jsonl"
        results = []
        persist_results(
            results, path=result_path, run_id=run_id, created_at=created_at,
            status="in_progress", failure_log_path=failure_log_path,
        )
        print("Starting new paired pilot:", result_path)
    else:
        payload = json.loads(resume_path.read_text(encoding="utf-8"))
        results = validate_resume_payload(payload)
        result_path = resume_path
        run_id = payload["run_id"]
        created_at = payload["created_at"]
        failure_log_path = Path(payload["failure_log_path"])
        print(f"Resuming {result_path} with {len(results)} completed runs.")

    completed_runs = {
        (row["trial"], row["scenario_id"], row["arm"]) for row in results
    }
    expected = TRIALS * len(BENCHMARK_SCENARIOS) * len(ARM_CONFIGS)
    rng = random.Random(RANDOM_SEED)
    try:
        for trial in range(1, TRIALS + 1):
            for scenario_id in BENCHMARK_SCENARIOS:
                order = ARM_CONFIGS.copy()
                rng.shuffle(order)
                for arm in order:
                    run_key = (trial, scenario_id, arm["name"])
                    if run_key in completed_runs:
                        print(f"skip completed trial={trial} scenario={scenario_id} arm={arm['name']}")
                        continue
                    print(f"trial={trial} scenario={scenario_id} arm={arm['name']}")
                    try:
                        result = run_scenario(arm, scenario_id, trial)
                    except BaseException as exc:
                        error_body = getattr(exc, "body", None)
                        error_body = error_body if isinstance(error_body, dict) else {}
                        append_failure_log(failure_log_path, {
                            "event": "paired_pilot_failure",
                            "trial": trial, "scenario_id": scenario_id,
                            "arm": arm["name"],
                            "exception_type": type(exc).__name__,
                            "message": str(exc),
                            "status_code": getattr(exc, "status_code", None),
                            "error_code": error_body.get("code"),
                            "request_id": getattr(exc, "request_id", None),
                            "solver_response_id": getattr(exc, "solver_response_id", None),
                            "solver_response_status": getattr(exc, "solver_response_status", None),
                            "solver_incomplete_reason": getattr(exc, "solver_incomplete_reason", None),
                            "traceback": traceback.format_exc(),
                        })
                        persist_results(
                            results, path=result_path, run_id=run_id,
                            created_at=created_at, status="interrupted",
                            failure_log_path=failure_log_path,
                        )
                        print("Completed results preserved:", result_path)
                        print("Failure log:", failure_log_path)
                        raise
                    results.append(result)
                    completed_runs.add(run_key)
                    status = "complete" if len(completed_runs) == expected else "in_progress"
                    persist_results(
                        results, path=result_path, run_id=run_id,
                        created_at=created_at, status=status,
                        failure_log_path=failure_log_path,
                    )
                    print(f"Saved completed runs={len(completed_runs)}/{expected}")
    finally:
        flush_traces()
    return results, result_path

pilot_results, pilot_artifact_path = (
    run_paired_pilot() if RUN_PAIRED_PILOT else ([], None)
)
print("Completed paired runs:", len(pilot_results))
print("Artifact:", pilot_artifact_path or "paid pilot disabled")


Starting new paired pilot: artifacts/02_retained_reasoning/results-20260829T054955277187Z.json
trial=1 scenario=POL2-401 arm=all_turns
Saved completed runs=1/30
trial=1 scenario=POL2-401 arm=current_turn
Saved completed runs=2/30
trial=1 scenario=POL2-402 arm=current_turn
Saved completed runs=3/30
trial=1 scenario=POL2-402 arm=all_turns
Saved completed runs=4/30
trial=1 scenario=POL2-403 arm=current_turn
Saved completed runs=5/30
trial=1 scenario=POL2-403 arm=all_turns
Saved completed runs=6/30
trial=1 scenario=POL2-404 arm=current_turn
Saved completed runs=7/30
trial=1 scenario=POL2-404 arm=all_turns
Saved completed runs=8/30
trial=1 scenario=POL2-405 arm=all_turns
Saved completed runs=9/30
trial=1 scenario=POL2-405 arm=current_turn
Saved completed runs=10/30
trial=2 scenario=POL2-401 arm=all_turns
Saved completed runs=11/30
trial=2 scenario=POL2-401 arm=current_turn
Saved completed runs=12/30
trial=2 scenario=POL2-402 arm=current_turn
Saved completed runs=13/30
trial=2 scenario=POL2-

## 8. Results: blind quality, strategy consistency, thinking, then cost

Learning and boundary accuracy describe exploration before labels are returned;
they are not pass/fail outcomes. Blind evaluation accuracy is primary. Strategy
consistency is supportive only when accuracy is acceptable—a consistently wrong
policy is not success. Thinking efficiency is evaluated on blind disposition
calls, and cost is interpreted only after quality gates pass.


In [7]:
def median_or_none(values):
    values = [value for value in values if value is not None]
    return median(values) if values else None

def phase_case_rows(run, phase):
    return [row for row in run["case_rows"] if row["phase"] == phase]

def summarize_results(results):
    summaries = []
    for arm_name in ("current_turn", "all_turns"):
        runs = [run for run in results if run["arm"] == arm_name]
        if not runs:
            continue
        learning = [row for run in runs for row in phase_case_rows(run, "learning")]
        boundary = [row for run in runs for row in phase_case_rows(run, "boundary")]
        evaluation = [row for run in runs for row in phase_case_rows(run, "evaluation")]
        successes = [
            run for run in runs
            if run["grade"]["evaluation_accuracy"] >= MINIMUM_EVALUATION_ACCURACY
            and run["grade"]["strategy_consistency"] >= MINIMUM_STRATEGY_CONSISTENCY
        ]
        summaries.append({
            "arm": arm_name,
            "runs": len(runs),
            "protocol_success_rate": sum(run["protocol_success"] for run in runs) / len(runs),
            "context_linkage_valid_rate": sum(run["context_ok"] and run["linkage_ok"] for run in runs) / len(runs),
            "mean_learning_accuracy": sum(row["accepted"] for row in learning) / len(learning),
            "mean_boundary_accuracy": sum(row["accepted"] for row in boundary) / len(boundary),
            "mean_evaluation_accuracy": sum(row["accepted"] for row in evaluation) / len(evaluation),
            "median_evaluation_errors": median(run["grade"]["evaluation_errors"] for run in runs),
            "median_strategy_consistency": median(run["grade"]["strategy_consistency"] for run in runs),
            "median_evaluation_decision_reasoning_tokens": median(
                row["decision_reasoning_tokens"] for row in evaluation
            ),
            "median_evaluation_decision_latency_ms": median(
                row["decision_latency_ms"] for row in evaluation
            ),
            "median_total_reasoning_tokens": median(run["total_reasoning_tokens"] for run in runs),
            "success_rate": len(successes) / len(runs),
            "standard_rate_cost_estimate_per_success_usd": (
                sum(run["standard_rate_cost_estimate_usd"] for run in runs) / len(successes)
                if successes else None
            ),
            "effective_service_tiers": sorted({
                tier for run in runs for tier in run["effective_service_tiers"]
            }),
        })
    return summaries

def paired_deltas(results):
    index = {(run["trial"], run["scenario_id"], run["arm"]): run for run in results}
    rows = []
    for trial in range(1, TRIALS + 1):
        for scenario_id in BENCHMARK_SCENARIOS:
            control = index.get((trial, scenario_id, "current_turn"))
            treatment = index.get((trial, scenario_id, "all_turns"))
            if not control or not treatment:
                continue
            control_eval = phase_case_rows(control, "evaluation")
            treatment_eval = phase_case_rows(treatment, "evaluation")
            rows.append({
                "trial": trial,
                "scenario_id": scenario_id,
                "evaluation_accuracy_delta": (
                    treatment["grade"]["evaluation_accuracy"]
                    - control["grade"]["evaluation_accuracy"]
                ),
                "evaluation_error_delta": (
                    treatment["grade"]["evaluation_errors"]
                    - control["grade"]["evaluation_errors"]
                ),
                "strategy_consistency_delta": (
                    treatment["grade"]["strategy_consistency"]
                    - control["grade"]["strategy_consistency"]
                ),
                "evaluation_decision_reasoning_delta": (
                    sum(row["decision_reasoning_tokens"] for row in treatment_eval)
                    - sum(row["decision_reasoning_tokens"] for row in control_eval)
                ),
                "evaluation_decision_latency_delta_ms": (
                    sum(row["decision_latency_ms"] for row in treatment_eval)
                    - sum(row["decision_latency_ms"] for row in control_eval)
                ),
                "standard_rate_cost_estimate_delta_usd": (
                    treatment["standard_rate_cost_estimate_usd"]
                    - control["standard_rate_cost_estimate_usd"]
                ),
            })
    return rows

summary_rows = summarize_results(pilot_results)
delta_rows = paired_deltas(pilot_results)
print(json.dumps(summary_rows, indent=2))
if delta_rows:
    print("Paired median deltas (all_turns - current_turn):")
    for key in (
        "evaluation_accuracy_delta", "evaluation_error_delta",
        "strategy_consistency_delta", "evaluation_decision_reasoning_delta",
        "evaluation_decision_latency_delta_ms",
        "standard_rate_cost_estimate_delta_usd",
    ):
        print(f"  {key}: {median_or_none(row[key] for row in delta_rows)}")


[
  {
    "arm": "current_turn",
    "runs": 15,
    "protocol_success_rate": 1.0,
    "context_linkage_valid_rate": 1.0,
    "mean_learning_accuracy": 0.4533333333333333,
    "mean_boundary_accuracy": 0.5666666666666667,
    "mean_evaluation_accuracy": 0.9566666666666667,
    "median_evaluation_errors": 1,
    "median_strategy_consistency": 0.95,
    "median_evaluation_decision_reasoning_tokens": 111.0,
    "median_evaluation_decision_latency_ms": 3519.1,
    "median_total_reasoning_tokens": 5716,
    "success_rate": 1.0,
    "standard_rate_cost_estimate_per_success_usd": 0.45474443999999997,
    "effective_service_tiers": [
      "priority"
    ]
  },
  {
    "arm": "all_turns",
    "runs": 15,
    "protocol_success_rate": 1.0,
    "context_linkage_valid_rate": 1.0,
    "mean_learning_accuracy": 0.46,
    "mean_boundary_accuracy": 0.5833333333333334,
    "mean_evaluation_accuracy": 0.9666666666666667,
    "median_evaluation_errors": 1,
    "median_strategy_consistency": 0.95,
    "me

## 9. Predeclared interpretation gates

Claim a combined workload-specific retained-reasoning benefit only when:

1. every paired unit has valid protocol, context, and response linkage;
2. `all_turns` mean blind accuracy is at least 70%, median consistency is at
   least 80%, and run-level success is non-inferior;
3. paired accuracy improves, errors decrease, and consistency does not regress;
4. evaluation decision reasoning decreases;
5. cost is interpreted only after quality passes.

Both arms above 95% trigger a ceiling warning. Both below 50% trigger a floor
warning and require better labeled calibration without changing the 70% gate.


In [8]:
def build_interpretation(summary, deltas, results):
    expected_runs = TRIALS * len(BENCHMARK_SCENARIOS) * len(ARM_CONFIGS)
    expected_pairs = TRIALS * len(BENCHMARK_SCENARIOS)
    by_arm = {row["arm"]: row for row in summary}
    delta_keys = (
        "evaluation_accuracy_delta", "evaluation_error_delta",
        "strategy_consistency_delta", "evaluation_decision_reasoning_delta",
        "evaluation_decision_latency_delta_ms",
        "standard_rate_cost_estimate_delta_usd",
    )
    paired_medians = {
        key: median_or_none(row[key] for row in deltas) for key in delta_keys
    }
    complete = len(results) == expected_runs and len(deltas) == expected_pairs
    harness_valid = complete and all(
        run["protocol_success"] and run["context_ok"] and run["linkage_ok"]
        for run in results
    )
    if len(by_arm) == 2:
        control, treatment = by_arm["current_turn"], by_arm["all_turns"]
        safe_quality = (
            treatment["success_rate"] >= control["success_rate"]
            and treatment["mean_evaluation_accuracy"] >= MINIMUM_EVALUATION_ACCURACY
            and treatment["median_strategy_consistency"] >= MINIMUM_STRATEGY_CONSISTENCY
        )
        strategy_learning_signal = (
            (paired_medians["evaluation_accuracy_delta"] or 0) > 0
            and (paired_medians["evaluation_error_delta"] or 0) < 0
            and (paired_medians["strategy_consistency_delta"] or 0) >= 0
        )
        thinking_efficiency_signal = (
            (paired_medians["evaluation_decision_reasoning_delta"] or 0) < 0
        )
        latency_corroborates = (
            (paired_medians["evaluation_decision_latency_delta_ms"] or 0) < 0
        )
        ceiling_warning = (
            control["mean_evaluation_accuracy"] > 0.95
            and treatment["mean_evaluation_accuracy"] > 0.95
        )
        floor_warning = (
            control["mean_evaluation_accuracy"] < 0.50
            and treatment["mean_evaluation_accuracy"] < 0.50
        )
    else:
        safe_quality = strategy_learning_signal = thinking_efficiency_signal = False
        latency_corroborates = ceiling_warning = floor_warning = False

    claim_supported = all((
        complete, harness_valid, safe_quality, strategy_learning_signal,
        thinking_efficiency_signal, not ceiling_warning, not floor_warning,
    ))
    if not complete:
        conclusion = "Complete the paired pilot before interpreting retained reasoning."
    elif not harness_valid:
        conclusion = "Harness validity failed; do not interpret quality, speed, or cost."
    elif ceiling_warning:
        conclusion = "Ceiling warning: increase transfer distance before confirmation."
    elif floor_warning:
        conclusion = "Floor warning: improve labeled calibration without lowering the quality gate."
    elif not safe_quality:
        conclusion = "Quality gate failed; do not interpret speed or cost as a benefit."
    elif claim_supported:
        conclusion = "Combined signal: retained reasoning improves blind strategy transfer and reduces decision-time thinking."
    elif strategy_learning_signal:
        conclusion = "Blind strategy transfer improved, but decision-time reasoning did not decrease."
    elif thinking_efficiency_signal:
        conclusion = "Decision-time reasoning decreased without a blind strategy-learning improvement."
    else:
        conclusion = "Neither retained-reasoning effect is supported for this workload and configuration."
    return {
        "expected_run_count": expected_runs,
        "completed_run_count": len(results),
        "expected_pair_count": expected_pairs,
        "completed_pair_count": len(deltas),
        "gates": {
            "pilot_complete": complete,
            "harness_valid": harness_valid,
            "safe_quality": safe_quality,
            "strategy_learning_signal": strategy_learning_signal,
            "thinking_efficiency_signal": thinking_efficiency_signal,
            "latency_corroborates": latency_corroborates,
            "no_ceiling": not ceiling_warning,
            "no_floor": not floor_warning,
        },
        "paired_medians": paired_medians,
        "ceiling_warning": ceiling_warning,
        "floor_warning": floor_warning,
        "claim_supported": claim_supported,
        "conclusion": conclusion,
    }

def finalize_artifact(path, results, summary, deltas, interpretation):
    if path is None:
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    saved_results = validate_resume_payload(payload)
    if stable_json_sha256(saved_results) != stable_json_sha256(results):
        raise RuntimeError("In-memory and persisted results differ.")
    payload.update({
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "completed_run_count": len(results),
        "analysis_complete": (
            payload.get("status") == "complete"
            and interpretation["gates"]["pilot_complete"]
        ),
        "analysis_generated_at": datetime.now(timezone.utc).isoformat(),
        "summary_rows": summary,
        "paired_deltas": deltas,
        "interpretation": interpretation,
    })
    atomic_write_json(path, payload)
    return payload

interpretation_result = build_interpretation(summary_rows, delta_rows, pilot_results)
print(interpretation_result["conclusion"])
final_artifact = finalize_artifact(
    pilot_artifact_path, pilot_results, summary_rows, delta_rows,
    interpretation_result,
)
if final_artifact is not None:
    print("Final analysis saved:", pilot_artifact_path)
    print("analysis_complete:", final_artifact["analysis_complete"])


Ceiling warning: increase transfer distance before confirmation.
Final analysis saved: artifacts/02_retained_reasoning/results-20260829T054955277187Z.json
analysis_complete: True


## 10. Trace checks and references

For each paired run, verify:

- all calls after the first reference the immediately prior response;
- requested and effective `reasoning.context` match;
- the paired arms use the same shuffled phase order;
- learning and boundary outputs include `accepted_action` while evaluation
  outputs contain only `ticket_id` and `recorded`;
- requested Fast tier and the API-reported effective tier are saved;
- every response is completed before tool arguments are parsed;
- artifacts update after each complete unit and interrupted units have a JSONL
  failure record.

References:

- [GPT-5.6 model guidance](https://developers.openai.com/api/docs/guides/latest-model)
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)
- [Function calling](https://developers.openai.com/api/docs/guides/function-calling)
- [Fast processing](https://developers.openai.com/api/docs/guides/fast-processing)
